In [1]:
# Install packages (run once)
# Install required packages (Run this cell only if the packages are not already installed.)
%pip install -U langchain langchain-openai langchain-community langchain-classic pypdf faiss-cpu tiktoken python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_classic.chains import RetrievalQA
from dotenv import load_dotenv

/var/folders/b0/zz5v88pn37799pf8zd4dnljw0000gn/T/ipykernel_72753/1084748110.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
# Load all 3 PDFs into a single list of documents
pdf_files = [
    "data/ETL Modernization with AWS Glue.pdf",
    "data/aws-glue-best-practices-build-efficient-data-pipeline.pdf",
    "data/serverless-etl-aws-glue.pdf"
]

documents = []
for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    documents.extend(loader.load())

#Verify
print(f"Total pages loaded: {len(documents)}")


Total pages loaded: 83


In [4]:
#Clean the documents by removing blank pages and repeated spaces
cleaned_documents = []
for doc in documents:
    text = doc.page_content

    # Replace repeated spaces, tabs, and line breaks with single spaces
    text = " ".join(text.split())

    # Skip blank pages or pages with very little useful text
    if len(text) < 100:
        continue

    doc.page_content = text
    cleaned_documents.append(doc)

# print(f"Pages before cleaning: {len(documents)}")
# print(f"Pages after cleaning: {len(cleaned_documents)}")

#Create a text splitter to chunk the documents into smaller pieces
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(cleaned_documents)
total_chunks = len(chunks)

average_chunk_size = sum(len(chunk.page_content) for chunk in chunks) / total_chunks

print(f"Total chunks: {total_chunks}")
print(f"Average chunk size: {average_chunk_size:.2f} characters")


Total chunks: 157
Average chunk size: 724.93 characters


In [5]:
load_dotenv(override=True)

True

In [6]:
## 1. Create the embedding model.
## 2. Pass all document chunks to the embedding model.
## 3. Store the resulting vectors in FAISS.

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)
# This creates a FAISS index object IN MEMORY
# It's stored in your computer's RAM, not on Azure or any cloud
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

In [7]:
chat_ai = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

qa_chain = RetrievalQA.from_chain_type(
    llm=chat_ai,
    retriever=retriever,
    return_source_documents=True
)

In [8]:
query = "how do I schedule AWS Glue jobs?"
result = qa_chain.invoke({"query": query})

print("Answer:")
print(result["result"])

print("\nTop matching source:")
top_doc = result["source_documents"][0]
print(top_doc.page_content)

print("\nSource file:")
print(top_doc.metadata.get("source"))

Answer:
Short answer
- Use an AWS Glue trigger (Scheduled trigger) or an Amazon EventBridge (CloudWatch Events) schedule to start your job on a recurring timetable. You can also start jobs on-demand or chain jobs using conditional triggers or Glue Workflows.

How to do it (high level)
1. Console (quickest)
   - Open the AWS Glue console → Triggers → Add trigger.
   - Choose type = Scheduled.
   - Enter a schedule expression (cron or rate-style expression) and attach one or more jobs as the trigger actions.
   - Enable the trigger. The Glue service will start the job runs on that schedule.

2. EventBridge (preferred when you want centralized scheduling or cross-service rules)
   - Create an EventBridge rule with a schedule (cron or rate).
   - Configure the rule to target Glue by calling the StartJobRun API (you can choose Glue job as a built-in target in the console).
   - Enable the rule.

3. CLI / SDK / IaC
   - Use the AWS SDK or CLI to create triggers (CreateTrigger) or to create a